# Проверка нормализации и клиппинга

Этот notebook проверяет добавление `normalized_C1` ... `normalized_C4` после расчета критериев.

In [6]:
from importlib import import_module
from pathlib import Path
import sys

# Определяем корень проекта.
project_root = Path.cwd()
if not (project_root / "data" / "trade.xlsx").exists():
    project_root = project_root.parent

# Добавляем корень проекта в пути импорта.
sys.path.insert(0, str(project_root))

loader = import_module("src.1_data_loader.loader")
preprocessing = import_module("src.2_preprocessing.preprocessing")
indicators = import_module("src.3_indicators.indicators")
normalization = import_module("src.4_normalization.normalization")

In [7]:
# Получаем критерии за выбранный год.
calculation_year = 2025

raw_data = loader.load_excel_data(project_root / "data" / "trade.xlsx")
prepared_data = preprocessing.preprocess_trade_data(raw_data)
yearly_trade = preprocessing.make_yearly_trade_table(prepared_data)
country_import = preprocessing.make_country_import_table(prepared_data)
indicator_values = indicators.calculate_indicators(
    yearly_trade,
    country_import,
    calculation_year,
)

In [8]:
# Нормализуем критерии с выбранным режимом клиппинга.
clipping_mode = "1-99"
normalized_values = normalization.normalize_indicators(indicator_values, clipping_mode)
normalized_values.head(20)

,TNVED,Year,Import,Export,C1,C2,C3,C4,normalized_C1,normalized_C2,normalized_C3,normalized_C4
0,841810,2025,2.084822e+08,18947441.75,1.498659e-02,-0.148730,0.916689,0.373150,0.420165,0.563135,0.912325,0.247527
1,841821,2025,4.494547e+07,296556.92,3.230872e-03,0.054670,0.993445,0.636188,0.090579,0.586585,0.993104,0.563279
2,841829,2025,4.078745e+06,968981.47,2.931976e-04,-0.192249,0.808036,0.905747,0.008217,0.558118,0.797977,0.886858
3,841830,2025,3.347578e+07,2247578.73,2.406382e-03,-0.160596,0.937084,0.675522,0.067463,0.561767,0.933788,0.610495
4,841840,2025,1.886445e+07,1048776.96,1.356057e-03,-0.155212,0.947333,0.305240,0.038016,0.562388,0.944574,0.166008
5,841850,2025,3.413444e+07,6095472.72,2.453729e-03,0.156111,0.848484,0.528235,0.068790,0.598280,0.840545,0.433691
6,841861,2025,4.227753e+06,43858.43,3.039089e-04,0.072806,0.989732,0.469133,0.008517,0.588676,0.989196,0.362746
7,841869,2025,8.253162e+07,12423519.11,5.932725e-03,-0.261416,0.869164,0.598667,0.166329,0.550144,0.862309,0.518239
8,841990,2025,1.629126e+08,4874260.67,1.171085e-02,0.622306,0.970950,0.871930,0.328326,0.652028,0.969429,0.846265
9,842211,2025,1.094512e+08,290074.48,7.867817e-03,-0.395526,0.997357,0.577024,0.220581,0.534682,0.997220,0.492258


In [9]:
# Проверяем структуру и диапазон нормализованных значений.
normalized_columns = ["normalized_C1", "normalized_C2", "normalized_C3", "normalized_C4"]

for column in normalized_columns:
    assert column in normalized_values.columns
    assert normalized_values[column].between(0, 1).all()

assert normalized_values[["C1", "C2", "C3", "C4"]].equals(indicator_values[["C1", "C2", "C3", "C4"]])

normalized_values[normalized_columns].describe()

,normalized_C1,normalized_C2,normalized_C3,normalized_C4
count,358.000000,358.000000,358.000000,358.000000
mean,0.070574,0.536197,0.855907,0.599707
std,0.154740,0.124958,0.221322,0.272714
min,0.000000,0.000000,0.000000,0.000000
25%,0.003066,0.492069,0.826838,0.374289
50%,0.013622,0.554067,0.947956,0.620513
75%,0.066586,0.587030,0.985409,0.836683
max,1.000000,1.000000,1.000000,1.000000


In [10]:
# Проверяем все поддерживаемые режимы клиппинга.
for mode in ["none", "1-99", "5-95"]:
    result = normalization.normalize_indicators(indicator_values, mode)
    for column in normalized_columns:
        assert result[column].between(0, 1).all()

print("Все режимы клиппинга работают корректно")

Все режимы клиппинга работают корректно
